In [1]:
puts `ls -l raw-data`

total 9540
-rw-rw-r-- 1 osboxes osboxes   34777 Apr  7 13:53 Demokritos-KG-information.xlsx
-rw-rw-r-- 1 osboxes osboxes  402701 Apr  7 13:53 Disease-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes   82213 Apr  7 13:53 Disease-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes   19841 Apr  7 13:53 Disease-Gene triples.tsv
-rw-rw-r-- 1 osboxes osboxes  207331 Apr  7 13:53 Disease-Therapeutic_Area.tsv
-rw-rw-r-- 1 osboxes osboxes   82213 Apr  7 13:53 Drug-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes 7769112 Apr  7 13:53 Drug-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  111643 Apr  7 13:53 Drug-Drug_type.tsv
-rw-rw-r-- 1 osboxes osboxes   87477 Apr  7 13:53 Drug-Gene triples.tsv
drwxrwxr-x 2 osboxes osboxes    4096 Apr  7 15:55 Feb 2026
-rw-rw-r-- 1 osboxes osboxes  684226 Apr  7 13:53 Gene-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes   87477 Apr  7 13:53 Gene-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  127519 Apr  7 13:53 Gene-Gene triples.tsv
-rw-rw-r-- 1 osboxes osboxes   3057

In [15]:
puts `head -10  "./raw-data/Drug-Gene triples.tsv"`

Drug	Drug_id	RELATION	PROVENANCE	Gene	Gene_id
Lysine	C0024337	PART_OF	["36128717_fullText_114"]	LOR gene	C1416897
Lysine	C0024337	PART_OF	["28062662_fullText_7","22578804_fullText_7"]	GCDH gene	C1415009
Lysine	C0024337	PART_OF	["8170567_fullText_3","17895983_fullText_69"]	MT-TA gene	C1537998
Lysine	C0024337	COEXISTS_WITH	["37197954_fullText_3","37197954_abstract_3"]	GFAP gene	C1415042
Lysine	C0024337	COEXISTS_WITH	["24594605_fullText_73"]	GCDH gene	C1415009
Lysine	C0024337	COEXISTS_WITH	["25333616_fullText_90"]	Caspase Gene	C1332858
Lysine	C0024337	COEXISTS_WITH	["25416781_fullText_5"]	ETFB gene	C1414472
Lysine	C0024337	INHIBITS	["36128717_fullText_28"]	LOR gene	C1416897
Lysine	C0024337	INHIBITS	["36128717_fullText_10"]	AASS gene	C1424938


In [16]:
puts `head -10  "./raw-data/Gene-Drug triples.tsv"`

Gene	Gene_id	RELATION	PROVENANCE	Drug	Drug_id
Lysine	C0024337	PART_OF	["36128717_fullText_114"]	LOR gene	C1416897
Lysine	C0024337	PART_OF	["28062662_fullText_7","22578804_fullText_7"]	GCDH gene	C1415009
Lysine	C0024337	PART_OF	["8170567_fullText_3","17895983_fullText_69"]	MT-TA gene	C1537998
Lysine	C0024337	COEXISTS_WITH	["37197954_fullText_3","37197954_abstract_3"]	GFAP gene	C1415042
Lysine	C0024337	COEXISTS_WITH	["24594605_fullText_73"]	GCDH gene	C1415009
Lysine	C0024337	COEXISTS_WITH	["25333616_fullText_90"]	Caspase Gene	C1332858
Lysine	C0024337	COEXISTS_WITH	["25416781_fullText_5"]	ETFB gene	C1414472
Lysine	C0024337	INHIBITS	["36128717_fullText_28"]	LOR gene	C1416897
Lysine	C0024337	INHIBITS	["36128717_fullText_10"]	AASS gene	C1424938


In [2]:
puts `head -5 ./maps/drug-mappings.map`

demokritosid,xref,demokritos_label,pubchem_cid,IUPACname
C0613621,http://purl.bioontology.org/ontology/MESH/C030536,"2,2-dichloro-1,1-difluoroethyl difluoromethyl ether",https://pubchem.ncbi.nlm.nih.gov/compound/152803,"2,2-dichloro-1,1-difluoroethyl difluoromethyl ether"
C0042291,http://purl.bioontology.org/ontology/SNOMEDCT/387080000,Valproic Acid,https://pubchem.ncbi.nlm.nih.gov/compound/3121,Valproic acid
C0059747,http://purl.bioontology.org/ontology/MESH/C007650,ethyl acetate,https://pubchem.ncbi.nlm.nih.gov/compound/8857,ethyl acetate
C0059747,http://purl.bioontology.org/ontology/SNOMEDCT/83298009,ethyl acetate,https://pubchem.ncbi.nlm.nih.gov/compound/8857,Ethyl acetate


In [3]:
puts `head -2 ./maps/2026-demokritos-disease-mondo.map`

demokritos_umls,prefname,mondo
C0000774,gastrin secretion abnormality,http://purl.obolibrary.org/obo/MONDO_0001770


In [3]:
require 'linkeddata'
require 'csv'

graphing_errors = File.open('./graph/drug-disease-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
entity_relations = CSV.read('./raw-data/Drug-Diseasetriples.tsv', col_sep: "\t", headers: true)
drug_mappings = CSV.read('./maps/drug-mappings.map', headers: true)
disease_mappings = CSV.read('./maps/2026-demokritos-disease-mondo.map', headers: true)

# Create RDF graph
graph = RDF::Repository.new

failures = {}
# Process each entity relation
entity_relations.each do |row|
  # warn row.inspect
  drug_id = row['Drug_id']
  disease_id = row['Disease_id']
  
  # Find corresponding mappings
  drug = drug_mappings.find { |d| d['demokritosid'] == drug_id }
  disease = disease_mappings.find { |d| d['demokritos_umls'] == disease_id }
  
  unless drug
    next if failures[drug_id]
    failures[drug_id] = 1
    warn "drug lookup failed #{drug_id}"
    graphing_errors.write "drug lookup failed #{drug_id}\n"
    next
  end
  unless disease
    next if failures[disease_id]
    failures[disease_id] = 1
    warn "disease lookup failed #{disease_id}"
    graphing_errors.write "disease lookup failed #{disease_id}\n"
    next
  end
  
  # Extract relevant IDs and labels
  # demokratisid,xref,demokratis_label,pubchem_cid,IUPACname
  # C0613621,http://purl.bioontology.org/ontology/MESH/C030536,"2,2-dichloro-1,1-difluoroethyl difluoromethyl ether",https://pubchem.ncbi.nlm.nih.gov/compound/152803,"2,2-dichloro-1,1-difluoroethyl%20difluoromethyl%20ether"
  pubchem_uri = RDF::URI.new(drug['pubchem_cid'])
  pubchem_type = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
  pubchem_label =  RDF::Literal.new(drug['xref'])
  pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
  # human_drug_label = RDF::Literal.new(drug['demokritos_label'])
  iupac_drug_label = RDF::Literal.new(drug['IUPACname'])
    
# demokritos_umls,prefname,mondo
# C0000774,gastrin secretion abnormality,http://purl.obolibrary.org/obo/MONDO_0001770
  mondo_uri = RDF::URI.new(disease['mondo'])
  mondo_type = RDF::URI.new("https://bioportal.bioontology.org/ontologies/MONDO")
  mondo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Disease")
  mondo_label =  RDF::Literal.new("MONDO Term")
#   orphanet = RDF::URI.new(disease['orpha'])
  disease_label = RDF::Literal.new(disease['prefname'])
  original_disease = RDF::Literal.new(disease_id)
  
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:#{drug_id}_#{disease_id}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
  
  # Add quads to graph using RDF::Statement
  graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], mondo_uri, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)
  
  # graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     human_drug_label, graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     iupac_drug_label, graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_type,          graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_core_type,             graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_type, RDFS.label,     RDF::Literal.new("PubChem"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_core_type, RDFS.label,     RDF::Literal.new("Drug"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{drug_id}"), graph_name: context_uri)
  
    
  graph << RDF::Statement.new(mondo_uri, RDFS.label, disease_label, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_type, RDFS.label, mondo_label, graph_name: context_uri)
#   graph << RDF::Statement.new(mondo_uri, SIMPATHIC['orphanet'], orphanet, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['original-id'], original_disease, graph_name: context_uri)

  graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("DEMOKRITOS"), graph_name: general_context)
#   graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-evidence'], RDF::Literal.new("TBD"), graph_name: context_uri)
end

# Write RDF to file in N-Quads format
File.open('./graph/demokritos_drug-disease.nq.large', 'w') do |f|
  RDF::Writer.for(:nquads).new(f) do |writer|
      warn "writing triples"
    writer << graph
  end
end
graphing_errors.close

puts "RDF quads written"

disease lookup failed C0524851
drug lookup failed C0026457
drug lookup failed C1567285
drug lookup failed C0723712
drug lookup failed C3541953
drug lookup failed C0008425
drug lookup failed C0006507
drug lookup failed C0591139
drug lookup failed C0002062
drug lookup failed C0024808
drug lookup failed C0060794
drug lookup failed C0056077
drug lookup failed C3492623
drug lookup failed C0042672
drug lookup failed C2825610
drug lookup failed C0041044
disease lookup failed C0011168
drug lookup failed C0070971
drug lookup failed C0699680
drug lookup failed C0082384
disease lookup failed C0013384
disease lookup failed C0679225
disease lookup failed C0742034
disease lookup failed C0694564
disease lookup failed C0262405
drug lookup failed C0017973
drug lookup failed C0073590
drug lookup failed C0772257
drug lookup failed C0242402
drug lookup failed C0595726
drug lookup failed C0001617
drug lookup failed C0045550
disease lookup failed C0442874
disease lookup failed C0042769
disease lookup failed

RDF quads written


In [18]:
graphing_errors.close